# Routing Robustness and Multiplicity Audit

This notebook audits the completed Qwen task/token/component routing experiment without loading the model or running any new interventions.

[实验设置与运行说明](../../../docs/experiment_notes/09_appendix_factor_routes/c45d480ab193d190.md)


In [ ]:
# Run this helper once. Every later code cell uses %%cell_progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = (
            int(match.group(1)), int(match.group(2)), match.group(3)
        )
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s",
                refresh=True,
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(
                f"complete | elapsed {elapsed:.0f}s", refresh=True
            )
        else:
            bar.set_postfix_str(
                f"failed | elapsed {elapsed:.0f}s", refresh=True
            )
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 12 tracked cells.')


## 1. Locate and lock the completed experiment

The prior candidate is read from the discovery decision file; this audit does not reselect a layer or token scope before testing robustness.


In [ ]:
%%cell_progress 1/12 Mount Drive and locate completed routing results
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque result artifacts')
from pathlib import Path

PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'results' / 'Qwen__Qwen3.5-9B').is_dir()
    ),
    None,
)
assert PROJECT_ROOT is not None, 'Could not find VisualTorque/results on Drive.'

MODEL_SLUG = 'Qwen__Qwen3.5-9B'
ROUTING_ROOT = (
    PROJECT_ROOT / 'results' / MODEL_SLUG
    / 'task_token_component_routing_v1'
)
SOURCE_ANALYSIS_DIR = ROUTING_ROOT / 'analysis'
AUDIT_DIR = SOURCE_ANALYSIS_DIR / 'robustness_audit_v1'
FIGURE_DIR = AUDIT_DIR / 'figures'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_PATHS = {
    'patch_records': SOURCE_ANALYSIS_DIR / 'patch_records.csv',
    'task_baselines': SOURCE_ANALYSIS_DIR / 'task_baselines.csv',
    'head_candidates': SOURCE_ANALYSIS_DIR / 'head_search_candidates.csv',
    'capture_config': ROUTING_ROOT / 'capture_config.json',
    'protocol_lock': ROUTING_ROOT / 'protocol_lock.json',
    'pair_manifest': ROUTING_ROOT / 'pair_manifest.jsonl',
}
missing = [str(path) for path in REQUIRED_PATHS.values() if not path.is_file()]
assert not missing, f'Missing completed routing artifacts: {missing}'
print('Project:', PROJECT_ROOT)
print('Routing results:', ROUTING_ROOT)
print('Audit output:', AUDIT_DIR)


In [ ]:
%%cell_progress 2/12 Load, validate, and freeze the robustness audit
cell_stage('Loading raw pair-level analysis tables')
import hashlib
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 20260824
BOOTSTRAP_REPLICATES = 4000
PERMUTATION_REPLICATES = 10000
ALPHA = 0.05
EXPECTED_PAIRS = 64
EXPECTED_PATCH_RECORDS = 42148

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)

patch_frame = pd.read_csv(REQUIRED_PATHS['patch_records'])
baseline_frame = pd.read_csv(REQUIRED_PATHS['task_baselines'])
prior_candidates = pd.read_csv(REQUIRED_PATHS['head_candidates'])
capture_config = json.loads(
    REQUIRED_PATHS['capture_config'].read_text(encoding='utf-8')
)
protocol_lock = json.loads(
    REQUIRED_PATHS['protocol_lock'].read_text(encoding='utf-8')
)
pair_manifest = pd.read_json(
    REQUIRED_PATHS['pair_manifest'], lines=True
)

assert len(patch_frame) == EXPECTED_PATCH_RECORDS
assert patch_frame['pair_id'].nunique() == EXPECTED_PAIRS
assert baseline_frame['pair_id'].nunique() == EXPECTED_PAIRS
assert len(pair_manifest) == EXPECTED_PAIRS
assert set(patch_frame['analysis_family'].unique()) == {
    'latent_task_route', 'individual_token_scan', 'component_patch'
}
assert patch_frame.loc[
    patch_frame['analysis_family'].isin([
        'individual_token_scan', 'component_patch'
    ]), 'valid_recovery'
].all()
assert len(prior_candidates) == 1, (
    'This audit expects the one candidate frozen by the completed run; '
    f'found {len(prior_candidates)}.'
)

frozen_candidate = {
    'scope': str(prior_candidates.iloc[0]['scope']),
    'layer_index': int(prior_candidates.iloc[0]['layer_index']),
    'component_type': 'attention_update',
}
token_mixer_kinds = capture_config['identity']['token_mixer_kinds']
frozen_candidate['module_kind'] = token_mixer_kinds.get(
    str(frozen_candidate['layer_index']),
    token_mixer_kinds.get(frozen_candidate['layer_index']),
)

audit_identity = {
    'schema_version': 'routing-robustness-audit-1.0',
    'source_patch_records_sha256': sha256_file(
        REQUIRED_PATHS['patch_records']
    ),
    'source_baselines_sha256': sha256_file(
        REQUIRED_PATHS['task_baselines']
    ),
    'source_capture_config_sha256': sha256_file(
        REQUIRED_PATHS['capture_config']
    ),
    'frozen_candidate': frozen_candidate,
    'alpha': ALPHA,
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'permutation_replicates': PERMUTATION_REPLICATES,
    'task_test': (
        'edit-type-blocked changed-minus-unchanged mean contrast; '
        'one-sided blocked permutation; BH across 96 route cells'
    ),
    'paired_family_test': (
        'pair-level one-sided sign-flip t statistic; BH FDR and '
        'Westfall-Young max-T within each frozen family'
    ),
    'candidate_gate': {
        'overall': (
            'both candidate contrasts require BH q<0.05 and max-T p<0.05'
        ),
        'stability': (
            'both contrasts positive in both deterministic halves and '
            'in at least three of four edit types'
        ),
        'successful_computation': (
            'both-correct outcome-changing subset has n>=12 and bootstrap '
            'lower bounds positive for both contrasts'
        ),
    },
    'seed': SEED,
}
AUDIT_LOCK_PATH = AUDIT_DIR / 'audit_protocol.json'
if AUDIT_LOCK_PATH.is_file():
    previous = json.loads(AUDIT_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == audit_identity, (
        'Existing audit uses different inputs or rules. Use a new audit version.'
    )
else:
    write_json_atomic(AUDIT_LOCK_PATH, {
        'identity': audit_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

print('Patch records:', len(patch_frame))
print('Pairs:', patch_frame['pair_id'].nunique())
print('Frozen candidate:', frozen_candidate)
assert frozen_candidate == {
    'scope': 'options_tokens',
    'layer_index': 16,
    'component_type': 'attention_update',
    'module_kind': 'linear_attn',
}, frozen_candidate


## 2. Frozen statistical utilities

Bootstrap intervals resample pairs.


In [ ]:
%%cell_progress 3/12 Define bootstrap, blocked-permutation, and max-T tests
cell_stage('Defining deterministic statistical utilities')

def stable_seed(label):
    digest = hashlib.sha256(f'{SEED}:{label}'.encode('utf-8')).hexdigest()
    return int(digest[:8], 16)

def bh_adjust(p_values):
    values = np.asarray(p_values, dtype=float)
    adjusted = np.full(values.shape, np.nan, dtype=float)
    valid = np.isfinite(values)
    if not valid.any():
        return adjusted
    valid_values = values[valid]
    order = np.argsort(valid_values)
    ranked = valid_values[order]
    scaled = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    scaled = np.minimum.accumulate(scaled[::-1])[::-1]
    restored = np.empty_like(scaled)
    restored[order] = np.minimum(scaled, 1.0)
    adjusted[valid] = restored
    return adjusted

def bootstrap_mean(values, label, replicates=BOOTSTRAP_REPLICATES):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {
            'n': 0, 'mean': np.nan,
            'ci_low': np.nan, 'ci_high': np.nan,
        }
    rng = np.random.default_rng(stable_seed(label))
    counts = rng.multinomial(
        len(values), np.full(len(values), 1 / len(values)),
        size=replicates,
    )
    draws = counts @ values / len(values)
    return {
        'n': int(len(values)),
        'mean': float(values.mean()),
        'ci_low': float(np.quantile(draws, 0.025)),
        'ci_high': float(np.quantile(draws, 0.975)),
    }

def stratified_binary_contrast(
    values, changed, strata, label,
    bootstrap_replicates=BOOTSTRAP_REPLICATES,
    permutation_replicates=PERMUTATION_REPLICATES,
):
    values = np.asarray(values, dtype=float)
    changed = np.asarray(changed, dtype=bool)
    strata = np.asarray(strata).astype(str)
    finite = np.isfinite(values)
    values, changed, strata = (
        values[finite], changed[finite], strata[finite]
    )
    blocks = []
    for stratum in sorted(set(strata)):
        indices = np.where(strata == stratum)[0]
        positive = indices[changed[indices]]
        negative = indices[~changed[indices]]
        if len(positive) and len(negative):
            effective = len(positive) * len(negative) / len(indices)
            blocks.append((stratum, positive, negative, effective))
    if not blocks:
        return {
            'changed_n': int(changed.sum()),
            'unchanged_n': int((~changed).sum()),
            'included_strata': '', 'effect': np.nan,
            'ci_low': np.nan, 'ci_high': np.nan,
            'p_value': np.nan,
        }
    weights = np.asarray([block[3] for block in blocks], dtype=float)
    weights /= weights.sum()
    observed_parts = np.asarray([
        values[positive].mean() - values[negative].mean()
        for _, positive, negative, _ in blocks
    ])
    observed = float(weights @ observed_parts)
    rng = np.random.default_rng(stable_seed(label))
    bootstrap = np.zeros(bootstrap_replicates, dtype=float)
    permutation = np.zeros(permutation_replicates, dtype=float)
    for weight, (_, positive, negative, _) in zip(weights, blocks):
        positive_counts = rng.multinomial(
            len(positive), np.full(len(positive), 1 / len(positive)),
            size=bootstrap_replicates,
        )
        negative_counts = rng.multinomial(
            len(negative), np.full(len(negative), 1 / len(negative)),
            size=bootstrap_replicates,
        )
        bootstrap += weight * (
            positive_counts @ values[positive] / len(positive)
            - negative_counts @ values[negative] / len(negative)
        )

        block_indices = np.concatenate([positive, negative])
        block_values = values[block_indices]
        positive_count = len(positive)
        random_order = np.argsort(
            rng.random((permutation_replicates, len(block_indices))),
            axis=1,
        )
        selected = random_order[:, :positive_count]
        selected_sum = block_values[selected].sum(axis=1)
        total_sum = block_values.sum()
        permuted_difference = (
            selected_sum / positive_count
            - (total_sum - selected_sum)
            / (len(block_indices) - positive_count)
        )
        permutation += weight * permuted_difference
    return {
        'changed_n': int(changed.sum()),
        'unchanged_n': int((~changed).sum()),
        'included_strata': '|'.join(block[0] for block in blocks),
        'effect': observed,
        'ci_low': float(np.quantile(bootstrap, 0.025)),
        'ci_high': float(np.quantile(bootstrap, 0.975)),
        'p_value': float(
            (1 + np.sum(permutation >= observed))
            / (permutation_replicates + 1)
        ),
    }

def paired_family_statistics(records, value_matrix, family_name):
    values = np.asarray(value_matrix, dtype=float)
    assert values.ndim == 2
    assert values.shape[1] == len(records)
    assert np.isfinite(values).all()
    n_pairs, n_tests = values.shape
    means = values.mean(axis=0)
    standard_errors = values.std(axis=0, ddof=1) / np.sqrt(n_pairs)
    observed_t = means / np.maximum(standard_errors, 1e-12)

    rng = np.random.default_rng(stable_seed(family_name))
    signs = rng.choice(
        np.asarray([-1.0, 1.0]),
        size=(PERMUTATION_REPLICATES, n_pairs),
    )
    permuted_means = signs @ values / n_pairs
    sum_squares = np.square(values).sum(axis=0)
    permuted_variance = (
        sum_squares[None, :]
        - n_pairs * np.square(permuted_means)
    ) / max(n_pairs - 1, 1)
    permuted_se = np.sqrt(
        np.maximum(permuted_variance, 0) / n_pairs
    )
    permuted_t = permuted_means / np.maximum(permuted_se, 1e-12)
    raw_p = (
        1 + np.sum(permuted_t >= observed_t[None, :], axis=0)
    ) / (PERMUTATION_REPLICATES + 1)
    max_t = np.max(permuted_t, axis=1)
    max_t_p = (
        1 + np.sum(max_t[:, None] >= observed_t[None, :], axis=0)
    ) / (PERMUTATION_REPLICATES + 1)

    counts = rng.multinomial(
        n_pairs, np.full(n_pairs, 1 / n_pairs),
        size=BOOTSTRAP_REPLICATES,
    )
    bootstrap_means = counts @ values / n_pairs
    rows = []
    for index, record in enumerate(records):
        rows.append({
            **record,
            'n': int(n_pairs),
            'mean': float(means[index]),
            'ci_low': float(np.quantile(
                bootstrap_means[:, index], 0.025
            )),
            'ci_high': float(np.quantile(
                bootstrap_means[:, index], 0.975
            )),
            't_statistic': float(observed_t[index]),
            'p_value': float(raw_p[index]),
            'p_max_t': float(max_t_p[index]),
        })
    result = pd.DataFrame(rows)
    result['q_bh'] = bh_adjust(result['p_value'])
    result['passes_fdr_and_max_t'] = (
        (result['mean'] > 0)
        & (result['q_bh'] < ALPHA)
        & (result['p_max_t'] < ALPHA)
    )
    return result

print('Frozen statistical utilities ready.')


## 3. Competence and latent relevance

Recovery of a model's source logits is meaningful even when the model is wrong, but it describes a decision pathway rather than a successful physical computation.


In [ ]:
%%cell_progress 4/12 Audit baseline competence and sample support
cell_stage('Building task-level competence strata')
baseline_frame['competence_stratum'] = np.select(
    [
        baseline_frame['source_correct'] & baseline_frame['target_correct'],
        baseline_frame['source_correct'] ^ baseline_frame['target_correct'],
    ],
    ['both_correct', 'exactly_one_correct'],
    default='neither_correct',
)
competence_summary = baseline_frame.groupby('task', sort=False).agg(
    pairs=('pair_id', 'nunique'),
    changed_pairs=('task_value_changed', 'sum'),
    source_accuracy=('source_correct', 'mean'),
    target_accuracy=('target_correct', 'mean'),
    both_correct_rate=('both_correct', 'mean'),
    both_correct_n=('both_correct', 'sum'),
).reset_index()
competence_by_change = baseline_frame.groupby(
    ['task', 'task_value_changed', 'competence_stratum'], sort=False
).size().rename('n').reset_index()
competence_summary.to_csv(
    AUDIT_DIR / 'baseline_competence.csv', index=False
)
competence_by_change.to_csv(
    AUDIT_DIR / 'baseline_competence_by_change.csv', index=False
)
display(competence_summary)
print('Outcome competence strata:')
display(
    baseline_frame[baseline_frame['task'] == 'outcome']
    .groupby(['task_value_changed', 'competence_stratum'])
    .size().rename('n').reset_index()
)


In [ ]:
%%cell_progress 5/12 Test changed-minus-unchanged latent relevance
cell_stage('Running 96 edit-blocked relevance tests')
task_frame = patch_frame[
    (patch_frame['analysis_family'] == 'latent_task_route')
    & patch_frame['valid_recovery']
].copy()
task_cells = list(task_frame.groupby(
    ['task', 'scope', 'layer_index'], sort=True
))
task_relevance_rows = []
for keys, group in tqdm(
    task_cells, desc='Latent relevance tests', unit='cell',
    dynamic_ncols=True,
):
    pair_values = group.groupby('pair_id', as_index=False).agg(
        recovery=('recovery', 'mean'),
        task_value_changed=('task_value_changed', 'first'),
        edit_type=('edit_type', 'first'),
    )
    statistics = stratified_binary_contrast(
        pair_values['recovery'],
        pair_values['task_value_changed'],
        pair_values['edit_type'],
        label=f'task:{keys[0]}:{keys[1]}:{int(keys[2])}',
    )
    task_relevance_rows.append({
        'task': keys[0], 'scope': keys[1],
        'layer_index': int(keys[2]), **statistics,
    })
task_relevance_audit = pd.DataFrame(task_relevance_rows)
task_relevance_audit['q_bh'] = bh_adjust(
    task_relevance_audit['p_value']
)
task_relevance_audit['passes_fdr'] = (
    (task_relevance_audit['effect'] > 0)
    & (task_relevance_audit['q_bh'] < ALPHA)
)
task_relevance_audit.to_csv(
    AUDIT_DIR / 'task_relevance_blocked_audit.csv', index=False
)
print(
    'FDR-significant positive latent-relevance cells:',
    int(task_relevance_audit['passes_fdr'].sum()),
    '/', len(task_relevance_audit),
)
display(
    task_relevance_audit.sort_values(
        ['q_bh', 'effect'], ascending=[True, False]
    ).head(20)
)


## 4. Family-wise component and token audits

Raw 95% intervals are not used as a selection rule.


In [ ]:
%%cell_progress 6/12 Correct component contrasts for multiplicity
cell_stage('Constructing paired component contrast families')
component_frame = patch_frame[
    (patch_frame['analysis_family'] == 'component_patch')
    & patch_frame['valid_recovery']
].copy()
pair_ids = sorted(component_frame['pair_id'].unique())

control_records = []
control_values = []
for keys, group in component_frame.groupby(
    ['component_type', 'scope', 'layer_index'], sort=True
):
    pivot = group.pivot_table(
        index='pair_id', columns='donor_type',
        values='recovery', aggfunc='first',
    ).reindex(pair_ids)
    assert {'matched_source', 'unrelated_same_source_outcome'} <= set(
        pivot.columns
    )
    difference = (
        pivot['matched_source']
        - pivot['unrelated_same_source_outcome']
    )
    assert difference.notna().all()
    control_records.append({
        'component_type': keys[0], 'scope': keys[1],
        'layer_index': int(keys[2]),
        'contrast': 'matched_minus_unrelated_same_outcome',
    })
    control_values.append(difference.to_numpy())
component_control_audit = paired_family_statistics(
    control_records, np.column_stack(control_values),
    'component_matched_control_24',
)

difference_records = []
difference_values = []
matched = component_frame[
    component_frame['donor_type'] == 'matched_source'
]
for keys, group in matched.groupby(['scope', 'layer_index'], sort=True):
    pivot = group.pivot_table(
        index='pair_id', columns='component_type',
        values='recovery', aggfunc='first',
    ).reindex(pair_ids)
    assert {'attention_update', 'mlp_update'} <= set(pivot.columns)
    difference = pivot['attention_update'] - pivot['mlp_update']
    assert difference.notna().all()
    difference_records.append({
        'scope': keys[0], 'layer_index': int(keys[1]),
        'contrast': 'attention_minus_mlp',
    })
    difference_values.append(difference.to_numpy())
attention_mlp_audit = paired_family_statistics(
    difference_records, np.column_stack(difference_values),
    'attention_minus_mlp_12',
)

component_control_audit.to_csv(
    AUDIT_DIR / 'component_control_multiplicity_audit.csv', index=False
)
attention_mlp_audit.to_csv(
    AUDIT_DIR / 'attention_mlp_multiplicity_audit.csv', index=False
)
print('Corrected matched-control component signals:')
display(component_control_audit[
    component_control_audit['passes_fdr_and_max_t']
])
print('Corrected attention-minus-MLP signals:')
display(attention_mlp_audit[
    attention_mlp_audit['passes_fdr_and_max_t']
])


In [ ]:
%%cell_progress 7/12 Correct 120 individual-token tests
cell_stage('Building paired token donor contrasts')
token_frame = patch_frame[
    (patch_frame['analysis_family'] == 'individual_token_scan')
    & patch_frame['valid_recovery']
].copy()
control_layers = sorted(
    set(token_frame.loc[
        token_frame['donor_type'] == 'unrelated_same_source_outcome',
        'layer_index',
    ].astype(int))
)
token_records = []
token_values = []
controlled = token_frame[token_frame['layer_index'].isin(control_layers)]
for keys, group in controlled.groupby(
    [
        'layer_index', 'token_group', 'group_rank',
        'token_index', 'token_string',
    ],
    sort=True,
):
    pivot = group.pivot_table(
        index='pair_id', columns='donor_type',
        values='recovery', aggfunc='first',
    ).reindex(pair_ids)
    assert {'matched_source', 'unrelated_same_source_outcome'} <= set(
        pivot.columns
    )
    difference = (
        pivot['matched_source']
        - pivot['unrelated_same_source_outcome']
    )
    assert difference.notna().all()
    token_records.append({
        'layer_index': int(keys[0]), 'token_group': keys[1],
        'group_rank': int(keys[2]), 'token_index': int(keys[3]),
        'token_string': keys[4],
    })
    token_values.append(difference.to_numpy())
token_multiplicity_audit = paired_family_statistics(
    token_records, np.column_stack(token_values),
    'individual_token_matched_control_120',
)
assert len(token_multiplicity_audit) == 120
token_multiplicity_audit.to_csv(
    AUDIT_DIR / 'individual_token_multiplicity_audit.csv', index=False
)
corrected_tokens = token_multiplicity_audit[
    token_multiplicity_audit['passes_fdr_and_max_t']
].sort_values('mean', ascending=False)
print('Raw positive-p<.05 tokens:', int((
    (token_multiplicity_audit['mean'] > 0)
    & (token_multiplicity_audit['p_value'] < ALPHA)
).sum()))
print('Tokens passing FDR and max-T:', len(corrected_tokens))
display(
    token_multiplicity_audit.sort_values(
        ['p_max_t', 'q_bh', 'mean'], ascending=[True, True, False]
    ).head(20)
)


## 5. Frozen candidate: competence and stability

The candidate remains `layer 16 × options_tokens × attention_update`.


In [ ]:
%%cell_progress 8/12 Build the frozen candidate pair table
cell_stage('Aligning candidate donor and component contrasts by pair')
candidate_rows = component_frame[
    (component_frame['scope'] == frozen_candidate['scope'])
    & (
        component_frame['layer_index']
        == frozen_candidate['layer_index']
    )
].copy()
candidate_pivot = candidate_rows.pivot_table(
    index='pair_id',
    columns=['component_type', 'donor_type'],
    values='recovery',
    aggfunc='first',
).reindex(pair_ids)
required_columns = {
    ('attention_update', 'matched_source'),
    ('attention_update', 'unrelated_same_source_outcome'),
    ('mlp_update', 'matched_source'),
}
assert required_columns <= set(candidate_pivot.columns)
metadata = candidate_rows.groupby('pair_id', as_index=True).agg(
    edit_type=('edit_type', 'first'),
    strategy_group=('strategy_group', 'first'),
    outcome_changed=('task_value_changed', 'first'),
    source_correct=('source_correct', 'first'),
    target_correct=('target_correct', 'first'),
    both_correct=('both_correct', 'first'),
).reindex(pair_ids)
candidate_pairs = metadata.copy()
candidate_pairs['attention_control_gap'] = (
    candidate_pivot[('attention_update', 'matched_source')]
    - candidate_pivot[
        ('attention_update', 'unrelated_same_source_outcome')
    ]
)
candidate_pairs['attention_minus_mlp'] = (
    candidate_pivot[('attention_update', 'matched_source')]
    - candidate_pivot[('mlp_update', 'matched_source')]
)
candidate_pairs['competence_stratum'] = np.select(
    [
        candidate_pairs['source_correct']
        & candidate_pairs['target_correct'],
        candidate_pairs['source_correct']
        ^ candidate_pairs['target_correct'],
    ],
    ['both_correct', 'exactly_one_correct'],
    default='neither_correct',
)

candidate_pairs = candidate_pairs.reset_index()
candidate_pairs['split_key'] = candidate_pairs['pair_id'].map(
    lambda value: hashlib.sha256(
        f'{SEED}:half:{value}'.encode('utf-8')
    ).hexdigest()
)
candidate_pairs = candidate_pairs.sort_values(
    ['edit_type', 'both_correct', 'split_key']
)
candidate_pairs['audit_half'] = (
    candidate_pairs.groupby(['edit_type', 'both_correct'])
    .cumcount().mod(2).map({0: 'half_A', 1: 'half_B'})
)
candidate_pairs = candidate_pairs.sort_values('pair_id').reset_index(drop=True)
assert len(candidate_pairs) == EXPECTED_PAIRS
assert candidate_pairs[
    ['attention_control_gap', 'attention_minus_mlp']
].notna().all().all()
candidate_pairs.to_csv(
    AUDIT_DIR / 'frozen_candidate_pair_effects.csv', index=False
)
display(candidate_pairs.head())
print('Deterministic half sizes:')
display(candidate_pairs.groupby(
    ['audit_half', 'both_correct']
).size().rename('n').reset_index())


In [ ]:
%%cell_progress 9/12 Audit candidate correctness and cross-group stability
cell_stage('Bootstrapping frozen candidate subgroups')
candidate_summary_rows = []

def append_candidate_group(group_family, group_name, subset):
    for effect in ('attention_control_gap', 'attention_minus_mlp'):
        candidate_summary_rows.append({
            'group_family': group_family,
            'group_name': str(group_name),
            'effect': effect,
            **bootstrap_mean(
                subset[effect].to_numpy(),
                f'candidate:{group_family}:{group_name}:{effect}',
            ),
        })

append_candidate_group('overall', 'all_pairs', candidate_pairs)
for value, group in candidate_pairs.groupby('competence_stratum'):
    append_candidate_group('competence', value, group)
append_candidate_group(
    'competence_binary', 'both_correct',
    candidate_pairs[candidate_pairs['both_correct']],
)
append_candidate_group(
    'competence_binary', 'not_both_correct',
    candidate_pairs[~candidate_pairs['both_correct']],
)
append_candidate_group(
    'competence_change', 'both_correct_and_changed',
    candidate_pairs[
        candidate_pairs['both_correct']
        & candidate_pairs['outcome_changed']
    ],
)
append_candidate_group(
    'competence_change', 'both_correct_and_unchanged',
    candidate_pairs[
        candidate_pairs['both_correct']
        & ~candidate_pairs['outcome_changed']
    ],
)
append_candidate_group(
    'competence_change', 'not_both_correct_and_changed',
    candidate_pairs[
        ~candidate_pairs['both_correct']
        & candidate_pairs['outcome_changed']
    ],
)
for value, group in candidate_pairs.groupby('edit_type'):
    append_candidate_group('edit_type', value, group)
for value, group in candidate_pairs.groupby('audit_half'):
    append_candidate_group('audit_half', value, group)
for value, group in candidate_pairs.groupby('outcome_changed'):
    append_candidate_group('outcome_changed', str(bool(value)), group)

candidate_stability = pd.DataFrame(candidate_summary_rows)
candidate_stability.to_csv(
    AUDIT_DIR / 'frozen_candidate_stability.csv', index=False
)
print('Overall and competence-stratified candidate effects:')
display(candidate_stability[
    candidate_stability['group_family'].isin([
        'overall', 'competence', 'competence_binary',
        'competence_change',
    ])
])
print('Edit-type and split-half stability:')
display(candidate_stability[
    candidate_stability['group_family'].isin([
        'edit_type', 'audit_half'
    ])
])


## 6. Figures and frozen decision gate

The gate distinguishes a mechanism of successful computation from a reliable but competence-independent decision route.


In [ ]:
%%cell_progress 10/12 Render competence, specificity, and robustness figures
cell_stage('Rendering audit figures')
plt.style.use('seaborn-v0_8-whitegrid')

task_order = competence_summary['task'].tolist()
figure, axis = plt.subplots(figsize=(11, 5))
axis.bar(
    competence_summary['task'],
    competence_summary['both_correct_rate'],
    color='#2563eb', alpha=0.85,
)
axis.axhline(1 / 9, color='#dc2626', linestyle='--', linewidth=1,
             label='independent random-choice reference')
axis.set_ylim(0, 1)
axis.set_ylabel('source and target both correct')
axis.set_title('Mechanistic interpretation must be competence-stratified')
axis.tick_params(axis='x', rotation=35)
axis.legend(frameon=False)
figure.tight_layout()
figure.savefig(
    FIGURE_DIR / 'baseline_competence.png', dpi=180,
    bbox_inches='tight',
)
plt.show()

scopes = ['image_tokens', 'all_text_except_final', 'final_prompt_token']
tasks = competence_summary['task'].tolist()
layers = sorted(task_relevance_audit['layer_index'].unique())
figure, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)
for axis, scope in zip(axes, scopes):
    subset = task_relevance_audit[
        task_relevance_audit['scope'] == scope
    ]
    matrix = subset.pivot(
        index='task', columns='layer_index', values='effect'
    ).reindex(index=tasks, columns=layers)
    significant = subset.pivot(
        index='task', columns='layer_index', values='passes_fdr'
    ).reindex(index=tasks, columns=layers).fillna(False)
    image = axis.imshow(
        matrix.to_numpy(), aspect='auto', cmap='coolwarm',
        vmin=-1, vmax=1,
    )
    for row in range(len(tasks)):
        for column in range(len(layers)):
            if bool(significant.iloc[row, column]):
                axis.text(column, row, '*', ha='center', va='center',
                          color='black', fontsize=13)
    axis.set_xticks(range(len(layers)), layers)
    axis.set_yticks(range(len(tasks)), tasks)
    axis.set_title(scope)
    axis.set_xlabel('layer')
figure.colorbar(
    image, ax=axes, label='blocked changed − unchanged recovery',
    shrink=0.8,
)
figure.suptitle('Latent relevance after edit-type blocking and BH correction')
figure.savefig(
    FIGURE_DIR / 'task_relevance_corrected.png', dpi=180,
    bbox_inches='tight',
)
plt.show()

ordered_tokens = token_multiplicity_audit.sort_values(
    'mean', ascending=False
).reset_index(drop=True)
figure, axis = plt.subplots(figsize=(12, 5))
colors = np.where(
    ordered_tokens['passes_fdr_and_max_t'], '#dc2626', '#94a3b8'
)
axis.scatter(
    np.arange(len(ordered_tokens)), ordered_tokens['mean'],
    c=colors, s=22,
)
axis.axhline(0, color='black', linewidth=0.8)
axis.set_xlabel('120 token tests ordered by observed effect')
axis.set_ylabel('matched − same-outcome-control recovery')
axis.set_title('Individual-token audit: red passes both FDR and max-T')
figure.tight_layout()
figure.savefig(
    FIGURE_DIR / 'token_multiplicity_audit.png', dpi=180,
    bbox_inches='tight',
)
plt.show()

forest_groups = [
    ('overall', 'all_pairs'),
    ('competence_binary', 'both_correct'),
    ('competence_change', 'both_correct_and_changed'),
    ('competence_binary', 'not_both_correct'),
    ('audit_half', 'half_A'), ('audit_half', 'half_B'),
    *[('edit_type', value) for value in sorted(
        candidate_pairs['edit_type'].unique()
    )],
]
figure, axes = plt.subplots(1, 2, figsize=(13, 7), sharey=True)
for axis, effect in zip(
    axes, ('attention_control_gap', 'attention_minus_mlp')
):
    rows = []
    for family, name in forest_groups:
        row = candidate_stability[
            (candidate_stability['group_family'] == family)
            & (candidate_stability['group_name'] == str(name))
            & (candidate_stability['effect'] == effect)
        ]
        assert len(row) == 1
        rows.append(row.iloc[0])
    y = np.arange(len(rows))
    means = np.asarray([row['mean'] for row in rows])
    lower = np.asarray([row['ci_low'] for row in rows])
    upper = np.asarray([row['ci_high'] for row in rows])
    axis.errorbar(
        means, y, xerr=np.vstack([means - lower, upper - means]),
        fmt='o', color='#2563eb', capsize=3,
    )
    axis.axvline(0, color='black', linewidth=0.8)
    axis.set_yticks(
        y, [f'{family}: {name}' for family, name in forest_groups]
    )
    axis.invert_yaxis()
    axis.set_title(effect)
    axis.set_xlabel('pair-level mean effect with 95% bootstrap CI')
figure.suptitle(
    'Frozen layer-16 options linear-attention candidate stability'
)
figure.tight_layout()
figure.savefig(
    FIGURE_DIR / 'candidate_stability_forest.png', dpi=180,
    bbox_inches='tight',
)
plt.show()
print('Figures saved to:', FIGURE_DIR)


In [ ]:
%%cell_progress 11/12 Apply the frozen decision gate and write the report
cell_stage('Evaluating precommitted route-follow-up criteria')
candidate_control_test = component_control_audit[
    (component_control_audit['component_type'] == 'attention_update')
    & (component_control_audit['scope'] == frozen_candidate['scope'])
    & (
        component_control_audit['layer_index']
        == frozen_candidate['layer_index']
    )
]
candidate_difference_test = attention_mlp_audit[
    (attention_mlp_audit['scope'] == frozen_candidate['scope'])
    & (
        attention_mlp_audit['layer_index']
        == frozen_candidate['layer_index']
    )
]
assert len(candidate_control_test) == 1
assert len(candidate_difference_test) == 1
candidate_control_test = candidate_control_test.iloc[0]
candidate_difference_test = candidate_difference_test.iloc[0]

corrected_overall = bool(
    candidate_control_test['passes_fdr_and_max_t']
    and candidate_difference_test['passes_fdr_and_max_t']
)

def candidate_group_mean(family, name, effect):
    row = candidate_stability[
        (candidate_stability['group_family'] == family)
        & (candidate_stability['group_name'] == str(name))
        & (candidate_stability['effect'] == effect)
    ]
    assert len(row) == 1
    return row.iloc[0]

effects = ('attention_control_gap', 'attention_minus_mlp')
halves_positive = all(
    candidate_group_mean('audit_half', half, effect)['mean'] > 0
    for half in ('half_A', 'half_B')
    for effect in effects
)
edit_support = sum(
    all(
        candidate_group_mean('edit_type', edit, effect)['mean'] > 0
        for effect in effects
    )
    for edit in sorted(candidate_pairs['edit_type'].unique())
)
stable_across_groups = bool(halves_positive and edit_support >= 3)
successful_rows = {
    effect: candidate_group_mean(
        'competence_change', 'both_correct_and_changed', effect
    )
    for effect in effects
}
successful_computation_support = bool(all(
    row['n'] >= 12 and row['ci_low'] > 0
    for row in successful_rows.values()
))

if (
    corrected_overall
    and stable_across_groups
    and successful_computation_support
):
    decision_code = 'proceed_linear_attention_path_decomposition'
    interpretation = (
        'The frozen linear-attention candidate is multiplicity-corrected, '
        'stable, and present in successful computations.'
    )
    recommended_next = (
        'Decompose the layer-16 linear-attention module and path-patch '
        'image → layer-16 option tokens → late response boundary → output, '
        'with layer-15 full attention as a negative architectural control.'
    )
elif corrected_overall and stable_across_groups:
    decision_code = 'decision_route_requires_correctness_followup'
    interpretation = (
        'The candidate is a robust model-decision route, but current data do '
        'not establish it as a mechanism of successful physical computation.'
    )
    recommended_next = (
        'Collect or select more both-correct counterfactual pairs and contrast '
        'successful versus error pathways before internal linear-attention '
        'decomposition.'
    )
else:
    decision_code = 'no_targeted_component_search_yet'
    interpretation = (
        'The discovery candidate does not survive the full multiplicity and '
        'stability gate.'
    )
    recommended_next = (
        'Do not run a head or neuron search. Increase pair support or test '
        'grouped route interventions with a new frozen confirmatory split.'
    )

significant_task_cells = task_relevance_audit[
    task_relevance_audit['passes_fdr']
].sort_values(['q_bh', 'effect'])
corrected_tokens = token_multiplicity_audit[
    token_multiplicity_audit['passes_fdr_and_max_t']
].sort_values(['p_max_t', 'q_bh'])

decision = {
    'schema_version': 'routing-robustness-decision-1.0',
    'frozen_candidate': frozen_candidate,
    'candidate_tests': {
        'attention_matched_control': {
            key: float(candidate_control_test[key])
            for key in ('mean', 'ci_low', 'ci_high', 'p_value',
                        'q_bh', 'p_max_t')
        },
        'attention_minus_mlp': {
            key: float(candidate_difference_test[key])
            for key in ('mean', 'ci_low', 'ci_high', 'p_value',
                        'q_bh', 'p_max_t')
        },
    },
    'corrected_overall': corrected_overall,
    'halves_positive': halves_positive,
    'supporting_edit_types': int(edit_support),
    'successful_computation_support': successful_computation_support,
    'fdr_task_relevance_cells': int(len(significant_task_cells)),
    'corrected_individual_tokens': int(len(corrected_tokens)),
    'decision': decision_code,
    'interpretation': interpretation,
    'recommended_next': recommended_next,
}
write_json_atomic(AUDIT_DIR / 'robustness_decision.json', decision)

report_lines = [
    '# VisualTorque routing robustness audit', '',
    f"- Frozen candidate: layer {frozen_candidate['layer_index']}, "
    f"`{frozen_candidate['scope']}`, `{frozen_candidate['module_kind']}`.",
    f'- Pairs: {EXPECTED_PAIRS}; patch records: {EXPECTED_PATCH_RECORDS}.',
    f'- Multiplicity: BH FDR plus paired sign-flip max-T at α={ALPHA}.',
    '', '## Competence', '',
]
for _, row in competence_summary.iterrows():
    report_lines.append(
        f"- {row['task']}: both-correct {int(row['both_correct_n'])}/"
        f"{int(row['pairs'])} ({row['both_correct_rate']:.1%})."
    )
report_lines.extend([
    '', '## Latent relevance', '',
    f'- {len(significant_task_cells)}/{len(task_relevance_audit)} '
    'edit-blocked changed−unchanged cells pass BH FDR.',
])
for _, row in significant_task_cells.head(12).iterrows():
    report_lines.append(
        f"- {row['task']}, layer {int(row['layer_index'])}, "
        f"`{row['scope']}`: Δ={row['effect']:.3f} "
        f"[{row['ci_low']:.3f}, {row['ci_high']:.3f}], "
        f"q={row['q_bh']:.4f}."
    )
report_lines.extend([
    '', '## Multiplicity-corrected localization', '',
    f'- Individual tokens passing both corrections: {len(corrected_tokens)}/120.',
    '- Frozen candidate attention matched−control: '
    f"{candidate_control_test['mean']:.3f} "
    f"[{candidate_control_test['ci_low']:.3f}, "
    f"{candidate_control_test['ci_high']:.3f}], "
    f"q={candidate_control_test['q_bh']:.4f}, "
    f"max-T p={candidate_control_test['p_max_t']:.4f}.",
    '- Frozen candidate attention−MLP: '
    f"{candidate_difference_test['mean']:.3f} "
    f"[{candidate_difference_test['ci_low']:.3f}, "
    f"{candidate_difference_test['ci_high']:.3f}], "
    f"q={candidate_difference_test['q_bh']:.4f}, "
    f"max-T p={candidate_difference_test['p_max_t']:.4f}.",
    '', '## Stability and decision', '',
    f'- Both deterministic halves positive: {halves_positive}.',
    f'- Edit types supporting both contrasts: {edit_support}/4.',
    '- Both-correct outcome-changing support for both contrasts: '
    f'{successful_computation_support}.',
    f'- **Decision: `{decision_code}`.**',
    f'- Interpretation: {interpretation}',
    f'- Next: {recommended_next}',
    '',
    'This audit is post-discovery robustness analysis, not an independent '
    'confirmatory replication.',
])
REPORT_PATH = AUDIT_DIR / 'robustness_audit_report.md'
REPORT_PATH.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
print(REPORT_PATH.read_text(encoding='utf-8'))


In [ ]:
%%cell_progress 12/12 Verify and package all audit outputs
cell_stage('Hashing audit deliverables and checking read-only provenance')
expected_outputs = [
    'audit_protocol.json',
    'baseline_competence.csv',
    'baseline_competence_by_change.csv',
    'task_relevance_blocked_audit.csv',
    'component_control_multiplicity_audit.csv',
    'attention_mlp_multiplicity_audit.csv',
    'individual_token_multiplicity_audit.csv',
    'frozen_candidate_pair_effects.csv',
    'frozen_candidate_stability.csv',
    'robustness_decision.json',
    'robustness_audit_report.md',
]
expected_figures = [
    'baseline_competence.png',
    'task_relevance_corrected.png',
    'token_multiplicity_audit.png',
    'candidate_stability_forest.png',
]
for name in expected_outputs:
    assert (AUDIT_DIR / name).is_file(), name
for name in expected_figures:
    assert (FIGURE_DIR / name).is_file(), name

manifest = {
    'schema_version': 'routing-robustness-artifacts-1.0',
    'source_artifacts': {
        name: {
            'path': str(path), 'sha256': sha256_file(path)
        }
        for name, path in REQUIRED_PATHS.items()
    },
    'audit_artifacts': {
        name: {
            'path': str(AUDIT_DIR / name),
            'sha256': sha256_file(AUDIT_DIR / name),
        }
        for name in expected_outputs
    },
    'figures': {
        name: {
            'path': str(FIGURE_DIR / name),
            'sha256': sha256_file(FIGURE_DIR / name),
        }
        for name in expected_figures
    },
    'decision': decision,
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
}
write_json_atomic(AUDIT_DIR / 'audit_manifest.json', manifest)
print('Audit complete.')
print('Decision:', decision['decision'])
print('Recommended next:', decision['recommended_next'])
print('Report:', REPORT_PATH)
print('Manifest:', AUDIT_DIR / 'audit_manifest.json')
